In [7]:
import numpy as np
from langchain_community.document_loaders import UnstructuredPDFLoader
import re

pdf_filepath = '/Users/sinequanon/Desktop/Capstone/data/pdf_data/TEST_02.pdf'

loader = UnstructuredPDFLoader(pdf_filepath, mode='elements')
pages = loader.load()

for page in pages:
    if page.page_content.lower() == 'plan':
        print(page.page_content, page.metadata['coordinates']['points'])
    
    if page.page_content.lower() == 'assessment':
        print(page.page_content, page.metadata['coordinates']['points'])

Plan ((36.02, 44.019000000000005), (36.02, 54.91899999999998), (57.264100000000006, 54.91899999999998), (57.264100000000006, 44.019000000000005))
Assessment ((36.02, 395.519), (36.02, 406.419), (93.63740000000001, 406.419), (93.63740000000001, 395.519))
Plan ((36.02, 463.54900000000004), (36.02, 474.449), (57.264100000000006, 474.449), (57.264100000000006, 463.54900000000004))
Plan ((36.02, 687.4490000000001), (36.02, 698.3489999999999), (57.264100000000006, 698.3489999999999), (57.264100000000006, 687.4490000000001))
Plan ((36.02, 571.269), (36.02, 582.169), (57.264100000000006, 582.169), (57.264100000000006, 571.269))


In [ ]:
def extract_subjective_data(pages):
    extracted_pages = []
    min_page = 50000
    min_height = 50000.0

    # 'Plan' 또는 'Assessment'의 최소 페이지와 y좌표 찾기
    for page in pages:
        metadata = page.metadata
        points = metadata['coordinates']['points']
        text = page.page_content

        if (text.lower() == 'plan' or text.lower() == 'assessment') and points[0][0] == 36.02:
            if metadata['page_number'] < min_page or (metadata['page_number'] == min_page and points[0][1] < min_height):
                min_page = metadata['page_number']
                min_height = points[0][1]

    # 'Plan' 또는 'Assessment' 이전의 페이지 데이터 추출
    for page in pages:
        metadata = page.metadata
        points = metadata['coordinates']['points']
        page_num = metadata['page_number']
        y_point = points[0][1]

        if page_num < min_page or (page_num == min_page and y_point < min_height):
            extracted_pages.append(page)

    return extracted_pages


In [8]:
import pandas as pd

In [9]:
data = {
    "코드": ["B1000", "B1015", "c2148", "ANB026", "c2007", "ANB090", "c2011", "DIG022", "NEU002", "c2079", "DIG073"],
    "항목명": [
        "혈구검사(CBC_Procyte)_혈액", "급성염증반응검사(CRP)_혈액", "내복약_1일2회_20kg", "세픽심", 
        "추가약_항생제(세픽심)_1T", "마보플록사신, 마보실", "추가약_항생제(마보플록사신)_1T", 
        "파모티딘", "가바펜틴", "추가약_진통,진정제(가바펜틴)_1T", "스멕타(경구용액)"
    ],
    "수량": ["1 회", "1 회", "5 회", "615 mg", "6 회", "123 mg", "6 회", "61.5 mg", "1230 mg", "4 회", "100 mL"],
    "일투": [None, None, None, 2, None, 1, None, 2, 2, None, 2],
    "일수": [None, None, None, 5, None, 5, None, 5, 5, None, 1],
    "총투": [None, None, None, 10, None, 5, None, 10, 10, None, 2],
    "Route": [None, None, None, "PO", None, "PO", None, "PO", "PO", None, "PO"],
    "Dose": [None, None, None, "(5 mg/kg)", None, "(2 mg/kg)", None, "(0.5 mg/kg)", "(10 mg/kg)", None, "(4.07 mL/kg)"]
}

In [11]:
df = pd.DataFrame(data)

print(df)

        코드                   항목명       수량   일투   일수    총투 Route          Dose
0    B1000  혈구검사(CBC_Procyte)_혈액      1 회  NaN  NaN   NaN  None          None
1    B1015      급성염증반응검사(CRP)_혈액      1 회  NaN  NaN   NaN  None          None
2    c2148         내복약_1일2회_20kg      5 회  NaN  NaN   NaN  None          None
3   ANB026                   세픽심   615 mg  2.0  5.0  10.0    PO     (5 mg/kg)
4    c2007       추가약_항생제(세픽심)_1T      6 회  NaN  NaN   NaN  None          None
5   ANB090           마보플록사신, 마보실   123 mg  1.0  5.0   5.0    PO     (2 mg/kg)
6    c2011    추가약_항생제(마보플록사신)_1T      6 회  NaN  NaN   NaN  None          None
7   DIG022                  파모티딘  61.5 mg  2.0  5.0  10.0    PO   (0.5 mg/kg)
8   NEU002                  가바펜틴  1230 mg  2.0  5.0  10.0    PO    (10 mg/kg)
9    c2079   추가약_진통,진정제(가바펜틴)_1T      4 회  NaN  NaN   NaN  None          None
10  DIG073             스멕타(경구용액)   100 mL  2.0  1.0   2.0    PO  (4.07 mL/kg)


In [17]:
data2 = {
    "코드" : [],
    "항목명" : [],
    "수량" : [],
    "일투" : [],
    "일수" : [],
    "총투" : [],
    "Route" : [],
    "Dose" : [],
}

data2['코드'].append("B1000")
data2['항목명'].append(None)

data2

{'코드': ['B1000'],
 '항목명': [None],
 '수량': [],
 '일투': [],
 '일수': [],
 '총투': [],
 'Route': [],
 'Dose': []}

In [41]:
import numpy as np
from langchain_community.document_loaders import UnstructuredPDFLoader
import re

pdf_filepath = '/Users/sinequanon/Desktop/Capstone/data/pdf_data/TEST_02.pdf'

loader = UnstructuredPDFLoader(pdf_filepath, mode='elements')
pages = loader.load()

for page in pages:
    if page.page_content.lower() == '일수 총투':
        print(page.page_content, page.metadata['coordinates']['points'])

일수 총투 ((369.93, 67.32000000000005), (369.93, 76.22000000000003), (420.31999999999994, 76.22000000000003), (420.31999999999994, 67.32000000000005))
일수 총투 ((369.93, 486.85), (369.93, 495.75), (420.31999999999994, 495.75), (420.31999999999994, 486.85))
일수 총투 ((369.93, 710.75), (369.93, 719.65), (420.31999999999994, 719.65), (420.31999999999994, 710.75))
일수 총투 ((369.93, 594.5699999999999), (369.93, 603.47), (420.31999999999994, 603.47), (420.31999999999994, 594.5699999999999))


In [46]:
info = {
        'y_points' : [ '220', '222', '234', '300'],
        'pages' : [1,1,2,5]
}

for point, page_num in info.items():
    print(point, page_num)

y_points ['220', '222', '234', '300']
pages [1, 1, 2, 5]
